In [3]:
# ======================================
# SmartChat Insight
#  Módulo de Predicción y Recomendación
# ======================================

# Importar librerías
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

In [4]:

# -------------------------------
# 1. Cargar los archivos finales
# -------------------------------
clientes = pd.read_csv("../data/outputs/final/clientes_powerbi.csv", sep=";")
clientes_final = pd.read_csv("../data/outputs/final/clientes_final_powerbi.csv", sep=";")
productos = pd.read_csv("../data/outputs/final/productos_powerbi.csv", sep=";")
resumen = pd.read_csv("../data/outputs/final/resumen_clientes.csv", sep=";")

print("Archivos cargados correctamente")
clientes_final.head()

Archivos cargados correctamente


,user,mensajes,primer_contacto,ultimo_contacto,dias_desde_ultimo,estado,accion_recomendada,producto_preferido,menciones_producto_preferido
0,+1 (772) 800-8661,9,2025-10-25,2025-10-27,0,Frecuente,Mantener flujo de comunicación,ventana,2
1,+34 610 94 76 16,9,2025-10-22,2025-10-22,5,Frecuente,Mantener flujo de comunicación,baño,1
2,+51 924 523 653,6,2025-08-04,2025-08-05,83,Perdido,Campaña de reactivación,vidrio,1
3,+57 300 2258585,8,2025-09-18,2025-10-02,25,Inactivo reciente,Mensaje de seguimiento,pasamanos,1
4,+57 300 3542763,8,2025-09-25,2025-09-26,31,Inactivo reciente,Mensaje de seguimiento,puerta,1


In [9]:
# Convertir columnas de fecha
for col in ["primer_contacto", "ultimo_contacto"]:
    if col in clientes_final.columns:
        clientes_final[col] = pd.to_datetime(clientes_final[col], errors="coerce")

# Crear fecha de corte
fecha_corte = clientes_final["ultimo_contacto"].max().date()
print("Fecha de corte del reporte:", fecha_corte)


Fecha de corte del reporte: 2025-10-27


In [8]:
# Reglas base
REGLAS = {
    "Frecuente": {
        "accion": "Mantener flujo de comunicación",
        "probabilidad_conversion": 0.75,
        "dias_para_contactar": 7
    },
    "Inactivo reciente": {
        "accion": "Mensaje de seguimiento",
        "probabilidad_conversion": 0.45,
        "dias_para_contactar": 2
    },
    "Perdido": {
        "accion": "Campaña de reactivación",
        "probabilidad_conversion": 0.18,
        "dias_para_contactar": 1
    },
}


In [10]:
# Cliente nuevo = primer contacto en últimos 3 días
clientes_final["nuevo"] = clientes_final["primer_contacto"] >= (
    pd.to_datetime(fecha_corte) - pd.Timedelta(days=3)
)

# Asignar etiqueta "Nuevo"
clientes_final.loc[clientes_final["nuevo"], "estado"] = "Nuevo"

# Agregar reglas para "Nuevo"
REGLAS["Nuevo"] = {
    "accion": "Mensaje de bienvenida",
    "probabilidad_conversion": 0.90,
    "dias_para_contactar": 0
}


In [11]:
def recomendar(row):
    estado = row["estado"]
    ultimo = row["ultimo_contacto"]

    regla = REGLAS.get(estado, None)
    if regla is None:
        return pd.Series({
            "accion_recomendada": "Revisar",
            "probabilidad_conversion": np.nan,
            "fecha_recomendada_contacto": None
        })
    
    accion = regla["accion"]
    prob = regla["probabilidad_conversion"]
    dias_para_contactar = regla["dias_para_contactar"]

    fecha_recomendada = ultimo + pd.Timedelta(days=dias_para_contactar)

    return pd.Series({
        "accion_recomendada": accion,
        "probabilidad_conversion": prob,
        "fecha_recomendada_contacto": fecha_recomendada.date()
    })

reco = clientes_final.apply(recomendar, axis=1)
clientes_final = pd.concat([clientes_final, reco], axis=1)


In [12]:
def generar_mensaje(row):
    estado = row["estado"]
    nombre = row["user"] if "user" in row else "cliente"

    mensajes = {
        "Frecuente": f"Hola {nombre}, gracias por mantener el contacto 😊. ¿En qué podemos ayudarte hoy?",
        "Inactivo reciente": f"Hola {nombre}, hace unos días no conversamos. ¿Te sigo ayudando con tu pedido?",
        "Perdido": f"Hola {nombre}, tenemos novedades y promociones disponibles para ti. ¿Deseas verlas?",
        "Nuevo": f"¡Bienvenido {nombre}! Gracias por escribirnos 😊. ¿En qué podemos ayudarte hoy?"
    }

    return mensajes.get(estado, "Hola, ¿cómo podemos ayudarte?")

clientes_final["mensaje_sugerido"] = clientes_final.apply(generar_mensaje, axis=1)


In [13]:
clientes_final[[
    "user", "estado", "dias_desde_ultimo",
    "accion_recomendada", "probabilidad_conversion",
    "fecha_recomendada_contacto", "mensaje_sugerido"
]].head(10)


,user,estado,dias_desde_ultimo,accion_recomendada,accion_recomendada,probabilidad_conversion,fecha_recomendada_contacto,mensaje_sugerido
0,+1 (772) 800-8661,Nuevo,0,Mantener flujo de comunicación,Mensaje de bienvenida,0.90,2025-10-27,¡Bienvenido +1 (772) 800-8661! Gracias por esc...
1,+34 610 94 76 16,Frecuente,5,Mantener flujo de comunicación,Mantener flujo de comunicación,0.75,2025-10-29,"Hola +34 610 94 76 16, gracias por mantener el..."
2,+51 924 523 653,Perdido,83,Campaña de reactivación,Campaña de reactivación,0.18,2025-08-06,"Hola +51 924 523 653, tenemos novedades y prom..."
3,+57 300 2258585,Inactivo reciente,25,Mensaje de seguimiento,Mensaje de seguimiento,0.45,2025-10-04,"Hola +57 300 2258585, hace unos días no conver..."
4,+57 300 3542763,Inactivo reciente,31,Mensaje de seguimiento,Mensaje de seguimiento,0.45,2025-09-28,"Hola +57 300 3542763, hace unos días no conver..."
5,+57 300 3580743,Frecuente,12,Mantener flujo de comunicación,Mantener flujo de comunicación,0.75,2025-10-22,"Hola +57 300 3580743, gracias por mantener el ..."
6,+57 300 3861858,Frecuente,10,Mantener flujo de comunicación,Mantener flujo de comunicación,0.75,2025-10-24,"Hola +57 300 3861858, gracias por mantener el ..."
7,+57 300 5157021,Perdido,82,Campaña de reactivación,Campaña de reactivación,0.18,2025-08-07,"Hola +57 300 5157021, tenemos novedades y prom..."
8,+57 300 5215647,Perdido,59,Campaña de reactivación,Campaña de reactivación,0.18,2025-08-30,"Hola +57 300 5215647, tenemos novedades y prom..."
9,+57 300 5251823,Perdido,61,Campaña de reactivación,Campaña de reactivación,0.18,2025-08-28,"Hola +57 300 5251823, tenemos novedades y prom..."


In [14]:
output_path = "../data/outputs/final/acciones_recomendadas.csv"
clientes_final.to_csv(output_path, index=False, sep=";")

print("Archivo exportado exitosamente:", output_path)


Archivo exportado exitosamente: ../data/outputs/final/acciones_recomendadas.csv
